<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-12-ethics-and-responsible-ai/lesson-12.4-privacy-and-data-protection/notebooks/exercises-12.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 12.4 — Privacy & Data Protection
Netsetos GenAI Course v2.0 | Module 12: Ethics & Responsible AI

PII taxonomy, DPDP Act 2023, redaction, data residency, DSAR, logging, DP/FL, breach response.


In [ ]:
print('Module 12: Ethics & Responsible AI')
print('Lesson 12.4: Privacy & Data Protection')


## Ex 1: PII Taxonomy (Direct + Quasi)


In [ ]:
print('PII categories + Indian specifics:')
for kind, examples, sensitivity in [
    ('Direct identifier', 'name, email, phone, address',              'personal'),
    ('Government ID',      'Aadhaar, PAN, passport, voter ID',         'SENSITIVE personal'),
    ('Financial',          'account number, card, UPI ID',             'SENSITIVE'),
    ('Biometric',          'fingerprint, face embedding, iris',         'SENSITIVE'),
    ('Health',             'diagnosis, prescription, genetic',          'SENSITIVE'),
    ('Location',           'GPS, IP, device ID',                        'personal'),
    ('Quasi-identifier',   'age + zip + gender (re-identifiable)',     'personal when combined'),
]:
    print(f'  {kind:22s}: {examples:42s} | {sensitivity}')


## Ex 2: DPDP Act 2023 — 6 Obligations


In [ ]:
print('DPDP Act 2023 — the six obligations every AI team must implement:')
for n, (obligation, practical_impl) in enumerate([
    ('Notice',            'clear notice in 22 languages on data collection'),
    ('Consent',           'free, specific, informed, WITHDRAWABLE'),
    ('Purpose limitation','use only for stated purpose; no scope creep'),
    ('Data minimization', 'collect only what is needed'),
    ('Accuracy + retention','correct + delete after purpose complete'),
    ('Security',          'reasonable safeguards + breach notice to DPB'),
], 1):
    print(f'  {n}. {obligation:24s}: {practical_impl}')


## Ex 3: Presidio Scrub Pipeline


In [ ]:
print('Presidio redaction flow:')
for step, what, notes in [
    ('1. Analyze',    'detect PII entities in text',    'built-in + custom recognizers'),
    ('2. Anonymize',  'replace with tokens',             '<PERSON_1>, <AADHAAR_1>'),
    ('3. Send LLM',    'prompt is PII-free',              'zero PII leaves VPC'),
    ('4. De-anonymize','restore tokens in final output', 'only if needed for user'),
    ('5. Audit log',   'log redacted version + counts',   'DPDP evidence'),
]:
    print(f'  {step:14s}: {what:32s} | {notes}')


## Ex 4: Aadhaar Pattern + Checksum


In [ ]:
import re
# Aadhaar: 12 digits, Verhoeff checksum
VERHOEFF_D = [
    [0,1,2,3,4,5,6,7,8,9],[1,2,3,4,0,6,7,8,9,5],[2,3,4,0,1,7,8,9,5,6],
    [3,4,0,1,2,8,9,5,6,7],[4,0,1,2,3,9,5,6,7,8],[5,9,8,7,6,0,4,3,2,1],
    [6,5,9,8,7,1,0,4,3,2],[7,6,5,9,8,2,1,0,4,3],[8,7,6,5,9,3,2,1,0,4],[9,8,7,6,5,4,3,2,1,0]
]
VERHOEFF_P = [
    [0,1,2,3,4,5,6,7,8,9],[1,5,7,6,2,8,3,0,9,4],[5,8,0,3,7,9,6,1,4,2],
    [8,9,1,6,0,4,3,5,2,7],[9,4,5,3,1,2,6,8,7,0],[4,2,8,6,5,7,3,9,0,1],
    [2,7,9,3,8,0,6,4,1,5],[7,0,4,6,9,1,3,2,5,8]
]
def verhoeff_ok(n):
    c = 0
    for i, d in enumerate(reversed([int(x) for x in str(n)])):
        c = VERHOEFF_D[c][VERHOEFF_P[i % 8][d]]
    return c == 0

AADHAAR = re.compile(r'\b(\d{4}[\s-]?\d{4}[\s-]?\d{4})\b')
text = 'My Aadhaar is 2345-6789-0123, please assist.'
for m in AADHAAR.finditer(text):
    digits = re.sub(r'\D','', m.group(1))
    print(f'candidate {digits}: verhoeff={verhoeff_ok(digits)}')


## Ex 5: Data Residency Decision Matrix


In [ ]:
print('Data residency decisions by sector (India, 2026):')
for sector, regulator, residency_rule, region in [
    ('Fintech / banking',    'RBI',      'store+process in India',     'asia-south1 (Mumbai)'),
    ('Insurance',            'IRDAI',    'India preferred',             'asia-south1'),
    ('Securities',           'SEBI',     'India mandated for customer data','asia-south1'),
    ('Health',               'MoHFW / sectoral','pending ABDM rules',    'asia-south1'),
    ('Telecom',              'DoT',      'India mandated for CDR',      'asia-south1'),
    ('Government notified',  'DPDP',     'India mandated (2026 list)',  'asia-south1'),
    ('General consumer',     'DPDP only','no cross-border ban by default','flexible'),
]:
    print(f'  {sector:22s} | regulator: {regulator:14s} | {residency_rule:30s} | {region}')


## Ex 6: DSAR (Data Subject Access Request) Flow


In [ ]:
print('DSAR workflow (DPDP 30-day SLA typical):')
for step, action, system in [
    ('1. Authenticate', 'verify identity (OTP + account)',     'auth service'),
    ('2. Map',          'find user data across all stores',    'data catalog'),
    ('3. App DB',        'SELECT WHERE user_id = ?',            'Postgres'),
    ('4. Vector DB',     'tombstone by user_id tag',            'Pinecone / pgvector'),
    ('5. Langfuse',      'drop traces by session / tenant',     'observability'),
    ('6. Cache',         'invalidate by user_id hash',          'Redis'),
    ('7. Logs',           'redact or purge old-retention buckets','S3/GCS'),
    ('8. Return / Delete','deliver JSON or confirm deletion',     'DSAR endpoint'),
    ('9. Audit',          'log request + action',                 'audit store'),
]:
    print(f'  {step:22s}: {action:36s} | system: {system}')


## Ex 7: Breach Response Runbook


In [ ]:
print('LLM-specific breach types + response:')
for breach, detect_signal, mitigation in [
    ('System prompt leak',      'user sees "you are..."',           'rotate prompts, output filter'),
    ('Cross-tenant retrieval',  'docs from tenant A in tenant B',   'vector DB isolation fix, audit'),
    ('Training-data exfil',      'memorized PII in generation',     'filter verbatim, notify DPB'),
    ('Credential in prompt log', 'AWS key in trace',                 'rotate, redact, notify'),
    ('Agent tool data exfil',    'anomalous file_read pattern',      'isolate tenant, rotate, page'),
]:
    print(f'  {breach:28s}: detect via {detect_signal:36s} | fix: {mitigation}')
print()
print('SLA: RBI 6h (fintech), DPDP "within prescribed time", SEBI similar. Tabletop quarterly.')


---
## Recap
PII taxonomy: direct + quasi + sensitive subsets (Aadhaar/PAN/health/biometric).
DPDP 6 obligations: notice, consent, purpose limit, minimization, accuracy+retention, security.
Redact (Presidio + Aadhaar+PAN recognizers) before LLM call.
Residency per sector (RBI India-only for fintech).
DSAR in 30 days; map data before requests arrive.
Hash + redact + short retention + KMS for logs. Rehearsed breach response.
